# Week 08 — Home exercise 4: Ten files, one table

**Solution proposal.**

A loop over a folder, `concat`, and the same aggregation written three ways so that the cost of each
is visible.

In [1]:
import os

import numpy as np
import pandas as pd

## Task 1: read ten files into one table

Three things happen inside the loop, and only one of them is reading the file. The `ticker` column has
to be added **here**, while we still know which file the rows came from — once everything is
concatenated, that information is gone forever.

In [2]:
folder = "../data/stocks"

files = sorted(name for name in os.listdir(folder) if name.endswith(".csv"))

print(files)

['AAPL.csv', 'AMZN.csv', 'BABA.csv', 'FB.csv', 'GOOG.csv', 'JNJ.csv', 'JPM.csv', 'MSFT.csv', 'TSLA.csv', 'WMT.csv']


In [3]:
frames = []

for name in files:
    prices = pd.read_csv(os.path.join(folder, name))

    prices["Date"] = pd.to_datetime(prices["Date"])
    prices["ticker"] = name.split(".")[0]

    frames.append(prices)

stocks = pd.concat(frames).reset_index(drop=True)

print("rows:   ", len(stocks))
print("tickers:", stocks["ticker"].nunique())
stocks.head(3)

rows:    2520
tickers: 10


,Date,Open,High,Low,Close,Adj Close,Volume,ticker
0,2020-01-02,74.059998,75.150002,73.797501,75.087502,74.333511,135480400,AAPL
1,2020-01-03,74.287498,75.144997,74.125000,74.357498,73.610840,146322800,AAPL
2,2020-01-06,73.447502,74.989998,73.187500,74.949997,74.197395,118387200,AAPL


2 520 rows: ten companies times 252 trading days. The `reset_index(drop=True)` matters — without it
the table has ten separate runs of 0 to 251, and every `.loc` on it returns ten rows.

## Task 2: check it before trusting it

In [4]:
print("exact duplicate rows:        ", stocks.duplicated().sum())
print("repeated ticker-date keys:   ", stocks.duplicated(subset=["ticker", "Date"]).sum())
print("rows per ticker:")
print(stocks["ticker"].value_counts().head(3))

exact duplicate rows:         0
repeated ticker-date keys:    0
rows per ticker:
ticker
AAPL    252
AMZN    252
BABA    252
Name: count, dtype: int64


Both zero, and every ticker has the same 252 rows. The table is what it claims to be.

**Why both checks, when they agreed here?** Because they fail on different things.

- `duplicated()` catches the same *file* being read twice — a name listed twice, a cell re-run, a
  colleague's copy of January dropped into the folder. Every column matches, so the rows are exact
  copies and dropping them loses nothing.
- `duplicated(subset=["ticker", "Date"])` catches the same *company on the same day* appearing twice
  **with different prices**. That is not a copy of anything, so the first check says nothing at all.

The second finding something the first did not would be the more serious result. It would mean two
sources disagree about what Apple closed at on some given day — and no amount of dropping rows
answers that. You would have to find out which file is right, and probably ask whoever sent them.

## Task 3, version 1: with loops

No pandas grouping anywhere. The outer loop picks a ticker, the inner loop picks a month, and a filter
inside both narrows the table to one cell of the answer.

In [5]:
tickers = []
months = []
volumes = []

for ticker in sorted(stocks["ticker"].unique()):
    one_company = stocks[stocks["ticker"] == ticker]

    for month in range(1, 13):
        one_month = one_company[one_company["Date"].dt.month == month]

        tickers.append(ticker)
        months.append(month)
        volumes.append(one_month["Volume"].sum())

by_loop = pd.DataFrame({"ticker": tickers, "month": months, "volume": volumes})

print(by_loop.shape)
by_loop.head(3)

(120, 3)


,ticker,month,volume
0,AAPL,1,2934370400
1,AAPL,2,3019851200
2,AAPL,3,6280072400


Fifteen lines, three parallel lists that have to stay in step, and a filter that runs 120 times over
the whole table.

## Version 2: with `groupby`

In [6]:
by_groupby = (
    stocks
    .groupby(["ticker", stocks["Date"].dt.month])["Volume"]
    .sum()
    .reset_index()
)

by_groupby.columns = ["ticker", "month", "volume"]

print(by_groupby.shape)
by_groupby.head(3)

(120, 3)


,ticker,month,volume
0,AAPL,1,2934370400
1,AAPL,2,3019851200
2,AAPL,3,6280072400


`stocks["Date"].dt.month` is passed straight to `groupby` as a grouping key without ever becoming a
column — grouping by something you computed on the spot is allowed, and often tidier than adding a
column you only need once.

## Version 3: with `resample`

In [7]:
by_resample = (
    stocks
    .set_index("Date")
    .groupby("ticker")
    .resample("ME")["Volume"]
    .sum()
    .reset_index()
)

print(by_resample.shape)
by_resample.head(3)

(120, 3)


,ticker,Date,Volume
0,AAPL,2020-01-31,2934370400
1,AAPL,2020-02-29,3019851200
2,AAPL,2020-03-31,6280072400


`groupby("ticker").resample("ME")` groups first by company and then resamples each company's rows on
the calendar — which is what you want, and is the same "group first" rule as everywhere else this
week. Without the `groupby`, the ten companies would be summed together.

Note the month column: `2020-01-31`, a real date, rather than the integer `1`. That is the difference
the lecture made a fuss about, and it is why this version sorts and plots correctly while the other
two need care.

### Do all three agree?

In [8]:
by_resample["month"] = by_resample["Date"].dt.month

check = (
    by_loop
    .merge(by_groupby, on=["ticker", "month"], suffixes=("_loop", "_groupby"), validate="one_to_one")
    .merge(by_resample[["ticker", "month", "Volume"]], on=["ticker", "month"], validate="one_to_one")
)

print("rows:", len(check))
print("loop == groupby: ", (check["volume_loop"] == check["volume_groupby"]).all())
print("loop == resample:", (check["volume_loop"] == check["Volume"]).all())

rows: 120
loop == groupby:  True
loop == resample: True


All three identical on all 120 rows.

### Which would you write?

`groupby`, without hesitating — or `resample`, if the answer is going onto a chart.

The loop is not wrong, and it is worth having written once to see that `groupby` is not magic: it is
the loop, done for you. But count what it costs.

| | Lines | Places a mistake can hide |
|---|---|---|
| Loop | ~15 | Three lists that must stay in step; `range(1, 13)` (off by one at either end); a month with no rows silently contributing 0; the ticker list needing `sorted` for reproducibility |
| `groupby` | 6 | The grouping key |
| `resample` | 7 | Forgetting the `groupby`, which would sum all ten companies together |

The loop's specific hazard is worth naming: `range(1, 13)` **asserts that there are twelve months
with data**. It happens to be true here. On a dataset that started in March, the loop would report
January and February as zero traded volume — which is a lie, not a gap — while `groupby` would simply
not produce those rows and `resample` would produce them as empty. Three different behaviors, and only
the loop's is actively misleading.

## Task 4: the biggest ticker-months of 2020

In [9]:
by_groupby.sort_values("volume", ascending=False).head(5)

,ticker,month,volume
2,AAPL,3,6280072400
7,AAPL,8,4070623100
8,AAPL,9,3885767100
3,AAPL,4,3266123200
5,AAPL,6,3243375600


**All five are Apple** — March, August, September, April and June. Apple is simply the most heavily
traded of the ten by a wide margin, so it takes every place before any other ticker appears.

That is worth pausing on. The question asked for the biggest ticker-months, and the honest answer is
that this ranking is mostly a fact about which company is biggest, not about which months were
eventful. March is the crash and August is Apple's four-for-one stock split, both real; but a ranking
across companies of different sizes will always be led by the largest one. To compare *months*, you
would want each company's volume relative to its own typical level — which is a `transform`, and is
exactly the pattern from exercise 2.

### Things worth noticing

- **`sorted()` on the file list is not decoration.** `os.listdir` returns names in whatever order the
  operating system feels like, which can differ between your machine and someone else's. An analysis
  whose output depends on that is not reproducible, and the fix costs six characters.
- **The filename was the only place the ticker existed.** Reading a folder of files is nearly always
  also a job of recording where each row came from, and the moment to do it is inside the loop.
- **`.endswith(".csv")` is the cheap defense** against `.ipynb_checkpoints` and `.DS_Store`, both of
  which appear without being asked and neither of which is a CSV.

### What this notebook does NOT do

- It assumes all ten files have the same columns. `concat` would not complain if one of them did not —
  it would add the extra column and fill 2 268 rows with `NaN`. Checking `prices.columns` inside the
  loop would catch that.
- It never checks that each file covers the same dates. Ten files of 252 rows each is consistent with
  ten completely different years, and nothing here would notice.
- The `Volume` sums are compared with `==` on floats-turned-integers, which happens to be safe because
  volumes are whole numbers. On genuinely floating-point results, `np.allclose` is the right
  comparison — `==` on two floats computed different ways is a coin toss.